# Who breaks Indian news?

Bharat Lens pulls articles from about 35 wire services and newspapers and groups the same
story from different outlets together. That gives a decent dataset for questions like:

1. Which outlet gets a story out first, and how far behind are the others?
2. Are Indian papers actually slower than the wires?
3. Which outlets cover the same things?
4. Does each outlet have a beat?
5. When does news get published?

The data comes from the views in `drizzle/manual/0002_analytics_views.sql` (the app's
`/insights` page uses the same ones). Analysis code is in `coverage_analysis.py`, charts in
`charts.py`.

To run: set `DATABASE_URL` (or put it in `.env.local`) and run all cells. `ANALYSIS_DAYS`
changes the window.

In [ ]:
import os

import pandas as pd

import charts
import coverage_analysis as cv

pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 20)

# days to look back, 0 = all time
DAYS = int(os.environ.get("ANALYSIS_DAYS", "30")) or None
corpus = cv.load(days=DAYS)
articles, clusters = corpus.articles, corpus.clusters
print(f"Window: {corpus.window_label}")
print(f"{len(articles):,} articles · {articles.cluster_id.nunique():,} stories · {articles.source_id.nunique()} outlets")

## Checking the publish times

Most of this depends on publish times, and publishers set those themselves. The SQL view
labels each one:

- `ok`: usable
- `missing`: no time given
- `date-only`: exactly midnight UTC or IST, so really just a date
- `future`: later than when we found the article (probably a timezone bug)
- `stale`: more than a week before we found it (old page republished)

I don't use `discovered_at` for any of this because the ingest only runs once a day.

In [ ]:
articles.time_quality.value_counts(normalize=True).rename("share").to_frame()

In [ ]:
quality = cv.data_quality(articles)
charts.data_quality_chart(quality);
quality.head(10)

## Who's first

Only stories that at least two outlets ran (with usable times) count. If an outlet is
the only one with a story it wasn't really racing anyone. Ties count as first for both.

I used Wilson intervals for the first-rate since some outlets have small samples, and a
bootstrap for the median delay because the delays are really skewed. Outlets with fewer
than 10 shared stories are dropped.

In [ ]:
speed = cv.speed(articles)
charts.speed_chart(speed);
speed[["source_name", "source_kind", "contested", "first_rate", "first_rate_lo", "first_rate_hi",
       "median_lag_min", "median_lag_lo", "median_lag_hi"]]

## Wires vs Indian papers

For stories that both a wire and an Indian paper ran, I check whose earliest report came
first. If neither side were faster this would be 50/50, so a binomial test works.

This only looks at stories the papers picked up at all, which is a bit of a selection
effect.

In [ ]:
wire = cv.wire_vs_press(articles)
charts.head_start_chart(wire["head_start"]);
{k: v for k, v in wire.items() if k != "head_start"}

In [ ]:
dependence = cv.wire_dependence(articles)
charts.wire_dependence_chart(dependence);
dependence

## Overlap

Jaccard similarity between outlets: stories both ran divided by stories either ran.
Sorted with hierarchical clustering so similar outlets end up next to each other.

In [ ]:
overlap = cv.overlap_matrix(articles)
charts.overlap_heatmap(overlap);
cv.top_pairs(overlap, 10)

## Beats

Chi-square test on outlet vs category. With this many rows the p-value will be tiny almost
no matter what, so Cramér's V is the more useful number.

Location quotient = outlet's share of a category / overall share. The heatmap uses
log2(LQ) so 2x and 0.5x look equally strong.

In [ ]:
spec = cv.specialisation(articles)
print(f"chi2({spec['dof']}) = {spec['chi2']:,.0f}, p = {spec['p_value']:.2g}, Cramér's V = {spec['cramers_v']:.2f}")
print(f"cells with expected count < 5: {spec['sparse_cell_share']:.0%}")
charts.specialisation_heatmap(spec["lq"]);
spec["top"].head(10)

## Timing, reach and India vs world

Times are in IST.

In [ ]:
grid = cv.rhythm(articles)
charts.rhythm_heatmap(grid);

In [ ]:
reach = cv.reach(clusters)
charts.reach_chart(reach["distribution"]);
print(f"single-outlet stories: {reach['single_source_share']:.0%} · 3+ outlets: {reach['three_plus_share']:.0%}")
reach["widest"]

In [ ]:
trend = cv.scope_trend(clusters)
charts.scope_trend_chart(trend);
trend

## Summary

Same text `report.py` writes to `output/FINDINGS.md`.

In [ ]:
import argparse
from IPython.display import Markdown

import report

results = {
    "quality": quality, "speed": speed, "wire": wire, "dependence": dependence,
    "overlap": overlap, "spec": spec, "rhythm": grid, "reach": reach, "trend": trend,
}
text = report.findings(corpus, results, argparse.Namespace(days=DAYS or 0, min_sample=cv.MIN_SAMPLE))
Markdown("\n".join(line for line in text.splitlines() if not line.startswith("![](")))

## Caveats

- Publish times are whatever the publisher says. If an outlet rounds or backdates them,
  its numbers are off and I can't tell.
- A story is a group of similar headlines within 48 hours, so sometimes two events get
  merged or one gets split. Sharing a story doesn't mean copying.
- Feeds only include part of what each outlet publishes, and some sources go quiet for
  weeks.
- This shows what happens, not why.